# Mutual Fund Analytics Platform — Day 6: Advanced Analytics
**Bluestock Fintech Capstone Project**  
Author: Shahin Shafi | June 2026

Advanced risk analytics including VaR/CVaR, rolling Sharpe, cohort analysis, SIP continuity, fund recommender, and HHI concentration.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import Image, display
import sys

BASE_DIR   = Path('.')
DB_PATH    = BASE_DIR / 'bluestock_mf.db'
PROC_DIR   = BASE_DIR / 'data' / 'processed'
CHARTS_DIR = BASE_DIR / 'reports' / 'charts'
sys.path.insert(0, str(BASE_DIR))

conn = sqlite3.connect(DB_PATH)
nav = pd.read_sql("""
    SELECT n.amfi_code, n.date_id, n.nav, f.scheme_name,
           f.sub_category, f.risk_category, f.plan
    FROM fact_nav n JOIN dim_fund f ON n.amfi_code=f.amfi_code
""", conn)
nav['date'] = pd.to_datetime(nav['date_id'])
nav_wide = nav.pivot_table(index='date', columns='amfi_code', values='nav')
returns  = nav_wide.pct_change().dropna()
tx = pd.read_sql('SELECT * FROM fact_transactions', conn)
tx['date'] = pd.to_datetime(tx['transaction_date'])
perf = pd.read_sql('SELECT p.*, f.sub_category, f.plan, f.fund_house, f.min_sip_amount FROM fact_performance p JOIN dim_fund f ON p.amfi_code=f.amfi_code', conn)
print(f'Returns matrix: {returns.shape}')
print(f'Transactions: {len(tx):,}')

## Task 1: Historical VaR (95%) & CVaR
**Formula:**
- `VaR(95%) = 5th percentile of daily returns`
- `CVaR = mean of all returns below VaR threshold`
- Higher absolute value = higher risk

In [ ]:
var_df = pd.read_csv(PROC_DIR / 'var_cvar_report.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top10_risk = var_df.nsmallest(10, 'var_95_daily_pct').copy()
top10_risk['short'] = top10_risk['scheme_name'].apply(lambda x: str(x).split(' - ')[0][:22])
axes[0].barh(top10_risk['short'][::-1], top10_risk['var_95_daily_pct'].abs()[::-1],
             color='#E53935', alpha=0.85, edgecolor='white')
axes[0].set_xlabel('VaR 95% (Absolute %)')
axes[0].set_title('Top 10 Highest Risk Funds (VaR 95%)', fontsize=12, fontweight='bold')

axes[1].scatter(var_df['var_95_daily_pct'].abs(), var_df['cvar_95_daily_pct'].abs(),
                c='#1565C0', s=80, alpha=0.8, edgecolors='white')
axes[1].set_xlabel('VaR 95% (Absolute %)')
axes[1].set_ylabel('CVaR 95% (Absolute %)')
axes[1].set_title('VaR vs CVaR — All 40 Funds', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(CHARTS_DIR / '26_var_cvar.png', dpi=150)
plt.show()
print('VaR Report:')
print(var_df[['scheme_name','var_95_daily_pct','cvar_95_daily_pct']].head(10).to_string(index=False))

## Task 2: Rolling 90-Day Sharpe Ratio
Formula: `rolling_sharpe = rolling_mean(r - Rf) / rolling_std(r) * sqrt(252)`  
Shows how risk-adjusted performance evolves over time.

In [ ]:
display(Image(filename=str(CHARTS_DIR / '22_rolling_sharpe.png'), width=900))

## Task 3: Investor Cohort Analysis
Investors grouped by their first transaction year (2024 or 2025).  
Analyzes average SIP amount, total invested, and top fund preference per cohort.

In [ ]:
display(Image(filename=str(CHARTS_DIR / '24_cohort_analysis.png'), width=900))

tx['first_year'] = tx.groupby('investor_id')['date'].transform('min').dt.year
cohort = tx[tx['transaction_type']=='SIP'].groupby('first_year').agg(
    investors=('investor_id','nunique'),
    avg_sip=('amount_inr','mean'),
    total_cr=('amount_inr', lambda x: x.sum()/1e7)
).round(2)
print('Cohort Summary:')
print(cohort.to_string())

## Task 4: SIP Continuity Analysis
For investors with 6+ SIP transactions, compute the average gap between dates.  
**At-risk flag:** investors with avg gap > 35 days (missing monthly SIPs).

In [ ]:
display(Image(filename=str(CHARTS_DIR / '25_sip_continuity.png'), width=900))

sip = tx[tx['transaction_type']=='SIP'].sort_values(['investor_id','date'])
sip_counts = sip.groupby('investor_id').size()
eligible = sip_counts[sip_counts >= 6].index
sip_elig = sip[sip['investor_id'].isin(eligible)]
gaps = sip_elig.groupby('investor_id')['date'].apply(
    lambda x: x.sort_values().diff().dt.days.mean()).dropna()
at_risk = (gaps > 35).sum()
print(f'Eligible investors : {len(gaps):,}')
print(f'At-risk (gap>35d)  : {at_risk:,} ({at_risk/len(gaps)*100:.1f}%)')
print(f'Consistent         : {(gaps<=35).sum():,}')
print(f'Median gap         : {gaps.median():.1f} days')

## Task 5: Fund Recommender System
Rule-based recommender matching investor risk appetite to fund risk grades.  
Scoring: 35% Sharpe + 30% 3yr Return + 20% Low Expense + 15% Rating

In [ ]:
from scripts.recommender import recommend_funds, print_recommendation

for risk in ['Low', 'Moderate', 'High']:
    print_recommendation(risk)

## Task 6: Sector HHI Concentration
**HHI = Σ(weight_i²)** — higher = more concentrated in fewer sectors.  
HHI > 0.15 = High concentration | 0.08-0.15 = Moderate | < 0.08 = Diversified

In [ ]:
display(Image(filename=str(CHARTS_DIR / '23_hhi_concentration.png'), width=900))

port = pd.read_sql("""
    SELECT p.amfi_code, p.sector, p.weight_pct, f.scheme_name
    FROM fact_portfolio p JOIN dim_fund f ON p.amfi_code=f.amfi_code
    WHERE f.category='Equity'
""", conn)
hhi = port.groupby('amfi_code')['weight_pct'].apply(
    lambda w: sum((w/100)**2)).reset_index()
hhi.columns = ['amfi_code','hhi']
hhi['concentration'] = hhi['hhi'].apply(
    lambda x: 'High' if x>0.15 else 'Moderate' if x>0.08 else 'Low')
print(hhi.sort_values('hhi',ascending=False).head(10).to_string(index=False))

## 5 Advanced Insights

### Insight 1 — Small Cap funds carry the highest tail risk
Small Cap funds show VaR(95%) of -2.3% to -2.4% daily, meaning on their worst 5% of days investors can lose over 2.3% in a single day. CVaR extends to -3.0%, indicating severe tail losses. Liquid funds, by contrast, have VaR below -0.03% — nearly 100x safer on a daily basis. *(VaR Chart)*

### Insight 2 — 2024 cohort dominates investment volume
The 2024 investor cohort (4,624 investors) contributed ₹21.5 Cr in SIP investments versus only ₹0.23 Cr from the 2025 cohort (138 investors). The 2025 cohort shows a higher average SIP of ₹13,505 vs ₹10,997 — suggesting newer investors are starting with larger ticket sizes. *(Cohort Chart)*

### Insight 3 — 97.8% of SIP investors are at-risk of discontinuity
Among investors with 6+ SIP transactions, 1,332 out of 1,362 (97.8%) have an average gap exceeding 35 days between SIPs. The median gap is 64.7 days — nearly double the 30-day monthly SIP cycle — signaling widespread SIP skipping behavior that needs intervention. *(Continuity Chart)*

### Insight 4 — Axis Bluechip shows highest sector concentration
Axis Bluechip Fund has the highest HHI of 0.206 with IT as its dominant sector, meaning its portfolio is heavily concentrated in a single sector. This creates sector-specific risk. Investors seeking true diversification should prefer funds with HHI below 0.10. *(HHI Chart)*

### Insight 5 — Liquid funds are optimal for Low risk investors
For Low risk appetite, ICICI Pru Liquid Fund achieves a Sharpe ratio of 7.68 — exceptionally high because of near-zero volatility. However, 3-year returns are only 7.68%, barely beating inflation. Moderate risk investors can achieve 14%+ CAGR with HDFC/Mirae Large Cap funds while maintaining a Sharpe above 1.0. *(Recommender output)*


In [ ]:
conn.close()
charts = list(CHARTS_DIR.glob('2*.png'))
print(f'Day 6 Advanced Analytics Complete!')
print(f'Charts: {len(charts)}')
print('Outputs: var_cvar_report.csv, recommender.py, 4 charts')